# LSTM real-vs-fake classifier on HF-vs-HR (PolitiFact++ & GossipCop++)

Faithful port of `true-and-fake-news-lstm-accuracy-97-90.ipynb` (Keras LSTM, ~97.9% on the
Kaggle ISOT set), pointed at the LIFE **human-written** subset. **Task = fake-vs-real among
human news:** HF (human_fake) = **0 (fake)**, HR (human_true) = **1 (real)** — the same
Fake=0 / True=1 convention as the source notebook.

- Same pipeline: clean/lemmatize → Keras `Tokenizer` → `pad_sequences(150)` →
  `Embedding → LSTM(150) → GlobalMaxPool → Dense → softmax(2)`, Adam 1e-4, 15 epochs.
- **Methodological contrast:** HF-vs-HR is the task LIFE's fingerprint method used as a
  *negative control* (it found no signal — neither class is LLM-generated). A content-based LSTM
  can instead pick up topical/stylistic differences, so it may separate them where LIFE couldn't.
- **Caveats:** PolitiFact++ HF/HR is only **291 articles** → the LSTM will **overfit**; read the
  test numbers as noisy. GossipCop++ (**12,252**) is fine. Both are **1:2 fake:real**, so watch
  **fake(HF=0) recall** and the confusion matrix, not just accuracy. High accuracy may also
  reflect source/topic style rather than genuine deception (as on the ISOT benchmark itself).
- **GPU optional** (TF uses it automatically if present; speeds up GossipCop++).

In [1]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TF version: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path below resolves
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

PolitiFact++ found: True
GossipCop++  found: True


## Run the classifier

Each cell prints the class balance, per-epoch train/val accuracy, then a test
`classification_report` + confusion matrix. PolitiFact++ is seconds; GossipCop++ (~9.8k train,
15 epochs) is a few minutes (faster on GPU).

In [4]:
!python lstm_real_vs_fake_code/run_life_lstm.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

[PolitiFact++] 271 articles | fake HF(0)=93 real HR(1)=178
vocab size = 15178
2026-07-19 18:23:55.934609: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784485435.935734    3849 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0
Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.6204 - loss: 0.6843 - val_accuracy: 0.6545 - val_loss: 0.6869
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6435 - loss: 0.6781 - val_accuracy: 0.6545 - val_loss: 0.6845
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6574 - loss: 0.6710 - val_accuracy: 0.6545 - val_loss: 0.6822
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6713 - loss: 0.6678 - val_accuracy

In [5]:
!python lstm_real_vs_fake_code/run_life_lstm.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

[GossipCop++] 11081 articles | fake HF(0)=3590 real HR(1)=7491
vocab size = 58681
2026-07-19 18:25:05.252323: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784485505.253474    5158 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0
Epoch 1/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6746 - loss: 0.6444 - val_accuracy: 0.6761 - val_loss: 0.6529
Epoch 2/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.6764 - loss: 0.6227 - val_accuracy: 0.6761 - val_loss: 0.6354
Epoch 3/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.7564 - loss: 0.5220 - val_accuracy: 0.7866 - val_loss: 0.5058
Epoch 4/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8466 - loss: 0

## Notes
- **Task = fake-vs-real among human news** (HF=0 vs HR=1) — NOT AI-vs-Human and NOT LIFE's
  MF-vs-MR. To change the cut, edit `FILE_LABELS` in `run_life_lstm.py`.
- Faithful to the source notebook (maxlen 150, Embedding 100, LSTM 150, Adam 1e-4, 15 epochs);
  the only deviation is a **stratified** 80/20 split so PolitiFact++'s tiny test set keeps the
  1:2 ratio. No model checkpoints saved.
- Expect PolitiFact++ to hit near-perfect train accuracy with an unstable test score — that's the
  overfitting caveat, not a bug. GossipCop++ should be more stable.